In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')

import numpy as np
import pandas as pd

import config
from pipeline import data_prep, selection, plots

In [ ]:
dd = data_prep.load_disease_filtered()
print(f'Final: {dd.Z_dis.shape[0]} samples, {len(np.unique(dd.dis_pheno))} phenotypes')
print(pd.Series(dd.dis_pheno).value_counts().to_string())

In [ ]:
Z_hc, hc_names = data_prep.load_z_hc()
print(f'Z_hc: {Z_hc.shape}')

In [ ]:
plots.plot_zscore_outlier_hist(dd.Z_dis, dd.dis_pheno)

In [ ]:
all_results, SELECTORS = selection.run_selection(dd.Z_dis, Z_hc, dd.dis_pheno, dd.gene_names)

In [ ]:
global_summary = pd.DataFrame([
    dict(method=k,
         n_genes=v['n_genes'],
         macro_sil=v['macro_sil'],
         macro_mc_auc=v['macro_mc'],
         mean_lr_auc=v['per_pheno']['auc_logreg'].mean(),
         mean_rf_auc=v['per_pheno']['auc_rf'].mean(),
         elapsed_s=int(v['t']))
    for k, v in all_results.items()
]).set_index('method').sort_values('macro_mc_auc', ascending=False).round(3)

print('== Global Summary ==')
display(global_summary)

best = global_summary.index[0]
display(all_results[best]['per_pheno'].round(3))

In [ ]:
# method별 ROC(binary/multiclass) + clustermap/UMAP → CV_Results/Figures/
for key in SELECTORS:
    plots.plot_roc_curves(key, all_results, dd.dis_pheno)
    plots.plot_selection_overview(key, all_results, dd.Z_dis, dd.dis_pheno, dd.dis_names, dd.gene_names)